# **Silver Transformations**

In [1]:
# HappyBooking - Step 5: Silver Transformations
#
# We start from the Bronze batch table and progressively clean it.
# Step 1: load Bronze, and define a reusable placeholder-cleaning
# function that we'll apply broadly before doing column-specific
# (typed) cleaning on the critical columns.

from pyspark.sql import functions as F

df_bronze = spark.table("bronze_hotel_booking_batch")
print(f"Bronze row count: {df_bronze.count()}")
print(f"Bronze column count: {len(df_bronze.columns)}")

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 3, Finished, Available, Finished, False)

Bronze row count: 1050638
Bronze column count: 72


In [2]:
# Step 2: Generic placeholder cleaning, applied to every string column.
#
# We define a reusable expression that:
#   1. Trims leading/trailing whitespace
#   2. Converts known placeholder values (e.g. "???", "___", "--",
#      empty string, whitespace-only) into real NULLs
#   3. Strips leading "!!" noise we observed in several fields
#
# This runs BEFORE type-specific cleaning, so that by the time we
# cast columns to numeric/date types, "garbage" values are already
# NULL instead of un-castable strings.

PLACEHOLDER_VALUES = ["???", "___", "--", "", "N/A", "NULL", "null"]

def clean_placeholder(col_name):
    """
    Returns a Spark Column expression that trims whitespace, strips
    a leading '!!' noise pattern, and converts placeholder-like
    values to NULL for the given column name.
    """
    c = F.trim(F.col(col_name))
    c = F.regexp_replace(c, r"^!!+", "")   # strip leading "!!" noise
    c = F.regexp_replace(c, r"\.\.\.$", "")  # strip trailing "..." noise
    c = F.trim(c)
    return F.when(c.isin(PLACEHOLDER_VALUES), None).otherwise(c)


string_columns = [f.name for f in df_bronze.schema.fields if f.dataType.simpleString() == "string"]
print(f"Applying placeholder cleaning to {len(string_columns)} string columns")

df_step1 = df_bronze
for col_name in string_columns:
    df_step1 = df_step1.withColumn(col_name, clean_placeholder(col_name))

df_step1.select("city", "country", "hotel_facilities", "booking_source").show(10, truncate=30)

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 4, Finished, Available, Finished, False)

Applying placeholder cleaning to 72 string columns
+------------+--------------------+----------------+--------------+
|        city|             country|hotel_facilities|booking_source|
+------------+--------------------+----------------+--------------+
|     Ostrava|      Czech Republic|            NULL|   Booking.com|
|    Brisbane|           Australia|      Restaurant|          NULL|
|Buenos Aires|           Argentina|      Restaurant|  TravelAgency|
| Cluj-Napoca|             Romania|    Pool,Gym,Spa|  TravelAgency|
|      Venice|               Italy|    Pool,Gym,Spa|       Expedia|
|    Arequipa|                Peru|      Restaurant|       Expedia|
|Kuala Lumpur|            Malaysia|       Free WiFi|        Direct|
|      Prague|      Czech Republic|    Pool,Gym,Spa|       Expedia|
|       Dubai|United Arab Emirates|      Restaurant|  TravelAgency|
|     Mombasa|               Kenya|      Restaurant|   Booking.com|
+------------+--------------------+----------------+-------------

In [3]:
# Step 3: Type casting and value-range validation for critical columns.
#
# We handle three distinct problems here:
#   1. Word-numbers (e.g. "Five") that block a direct numeric cast
#   2. Sentinel/out-of-range numbers (e.g. 9999, -1) used as
#      de facto "missing value" markers in this dataset
#   3. Invalid date strings that don't correspond to real calendar
#      dates (e.g. "Jan 32, 2025", "0000-00-00", "2023-13-01")

WORD_TO_NUMBER = {
    "one": "1", "two": "2", "three": "3", "four": "4", "five": "5",
    "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
}

def clean_numeric(col_name, min_valid=0, max_valid=None):
    """
    Casts a string column to double, first replacing common spelled-out
    numbers with digits. Values outside [min_valid, max_valid] are
    treated as sentinel/garbage and set to NULL rather than kept as
    misleading outliers.
    """
    c = F.lower(F.trim(F.col(col_name)))
    for word, digit in WORD_TO_NUMBER.items():
        c = F.when(c == word, digit).otherwise(c)
    c = c.cast("double")

    if max_valid is not None:
        c = F.when((c < min_valid) | (c > max_valid), None).otherwise(c)
    else:
        c = F.when(c < min_valid, None).otherwise(c)

    return c


def clean_date(col_name):
    """
    Casts a string column to date using Spark's to_date, which
    returns NULL for strings that aren't valid calendar dates
    (e.g. "2023-13-01", "Jan 32, 2025") rather than raising an
    error. We also guard against the "0000-00-00" placeholder
    pattern explicitly, since some date libraries parse it as a
    valid-but-meaningless date.
    """
    c = F.col(col_name)
    c = F.when(c == "0000-00-00", None).otherwise(c)
    return F.to_date(c, "yyyy-MM-dd")


df_step2 = (
    df_step1
    .withColumn("total_rooms", clean_numeric("total_rooms", min_valid=1, max_valid=2000))
    .withColumn("star_rating", clean_numeric("star_rating", min_valid=1, max_valid=5))
    .withColumn("nights", clean_numeric("nights", min_valid=1, max_valid=90))
    .withColumn("adults", clean_numeric("adults", min_valid=0, max_valid=20))
    .withColumn("children", clean_numeric("children", min_valid=0, max_valid=20))
    .withColumn("infants", clean_numeric("infants", min_valid=0, max_valid=20))
    .withColumn("rooms_booked", clean_numeric("rooms_booked", min_valid=1, max_valid=50))
    .withColumn("total_price", clean_numeric("total_price", min_valid=0, max_valid=None))
    .withColumn("room_price", clean_numeric("room_price", min_valid=0, max_valid=None))
    .withColumn("tax_amount", clean_numeric("tax_amount", min_valid=0, max_valid=None))
    .withColumn("service_fee", clean_numeric("service_fee", min_valid=0, max_valid=None))
    .withColumn("paid_amount", clean_numeric("paid_amount", min_valid=0, max_valid=None))
    .withColumn("booking_date", clean_date("booking_date"))
    .withColumn("checkin_date", clean_date("checkin_date"))
    .withColumn("checkout_date", clean_date("checkout_date"))
)

df_step2.select(
    "total_rooms", "star_rating", "nights", "rooms_booked",
    "booking_date", "checkin_date", "checkout_date"
).show(10)

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 5, Finished, Available, Finished, False)

+-----------+-----------+------+------------+------------+------------+-------------+
|total_rooms|star_rating|nights|rooms_booked|booking_date|checkin_date|checkout_date|
+-----------+-----------+------+------------+------------+------------+-------------+
|      480.0|        3.0|   8.0|         1.0|  2024-03-27|        NULL|   2026-01-09|
|      291.0|        3.0|   6.0|         1.0|  2024-11-26|  2026-10-29|   2026-11-04|
|       28.0|        5.0|   7.0|         3.0|  2024-07-03|  2025-10-14|   2025-10-21|
|      208.0|        2.0|  12.0|         3.0|  2024-01-03|  2025-01-28|   2025-02-09|
|      390.0|        2.0|  10.0|         2.0|  2025-01-02|  2026-03-10|   2026-03-20|
|       91.0|        2.0|   8.0|         1.0|  2025-01-25|  2024-11-13|   2024-11-21|
|      396.0|        4.0|   8.0|         1.0|  2023-11-16|  2024-11-21|   2024-11-29|
|      432.0|        4.0|   1.0|         1.0|        NULL|  2024-11-23|   2024-11-24|
|      382.0|        3.0|  12.0|         1.0|  2025-04

In [4]:
# Step 4: Cross-field date validation.
#
# A booking's check-in date should not be before the date the
# booking was made, and check-out should not be before check-in.
# We don't null out the dates here - a flag preserves the original
# (suspicious) values while still letting downstream consumers
# filter on validity if they choose to.

df_step3 = df_step2.withColumn(
    "is_date_logic_valid",
    F.when(
        F.col("booking_date").isNull()
        | F.col("checkin_date").isNull()
        | F.col("checkout_date").isNull(),
        None,  # can't evaluate logic when a date is missing
    )
    .when(
        (F.col("checkin_date") < F.col("booking_date"))
        | (F.col("checkout_date") < F.col("checkin_date")),
        False,
    )
    .otherwise(True)
)

df_step3.groupBy("is_date_logic_valid").count().show()

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 6, Finished, Available, Finished, False)

+-------------------+------+
|is_date_logic_valid| count|
+-------------------+------+
|               NULL|114185|
|               true|813909|
|              false|122544|
+-------------------+------+



In [5]:
# Step 5: Standardize categorical/text fields - casing consistency
# and boolean normalization.
#
# We saw inconsistent casing throughout (e.g. "SOUTH KOREA" vs
# "South Africa", "credit card" vs "Credit Card"). We apply
# consistent title-casing to categorical text fields, and normalize
# boolean-like string columns into actual booleans.

def title_case(col_name):
    return F.initcap(F.trim(F.col(col_name)))


def clean_boolean(col_name):
    """
    Normalizes boolean-like strings (True/False in various casings,
    with possible whitespace noise already handled upstream) into
    real boolean type. Anything unrecognized becomes NULL rather
    than a guessed default.
    """
    c = F.lower(F.trim(F.col(col_name)))
    return (
        F.when(c == "true", True)
        .when(c == "false", False)
        .otherwise(None)
    )


df_step4 = (
    df_step3
    .withColumn("country", title_case("country"))
    .withColumn("city", title_case("city"))
    .withColumn("payment_status", title_case("payment_status"))
    .withColumn("payment_method", title_case("payment_method"))
    .withColumn("booking_status", title_case("booking_status"))
    .withColumn("is_cancelled", clean_boolean("is_cancelled"))
)

df_step4.select(
    "country", "city", "payment_method", "booking_status", "is_cancelled"
).show(10)

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 7, Finished, Available, Finished, False)

+--------------------+------------+--------------+--------------+------------+
|             country|        city|payment_method|booking_status|is_cancelled|
+--------------------+------------+--------------+--------------+------------+
|      Czech Republic|     Ostrava|        Paypal|     Confirmed|        true|
|           Australia|    Brisbane|    Debit Card|     Cancelled|        true|
|           Argentina|Buenos Aires|        Paypal|     Confirmed|        true|
|             Romania| Cluj-napoca|        Paypal|     Cancelled|        true|
|               Italy|      Venice|   Credit Card|     Confirmed|        true|
|                Peru|    Arequipa|        Paypal|     Confirmed|       false|
|            Malaysia|Kuala Lumpur|    Debit Card|       Pending|        true|
|      Czech Republic|      Prague|        Paypal|     Cancelled|       false|
|United Arab Emirates|       Dubai|    Debit Card|       Pending|        true|
|               Kenya|     Mombasa|          Cash|  

In [6]:
# Step 6: ID column validation and duplicate removal.
#
# ID columns (hotel_id, booking_id, customer_id) should never be
# placeholder/blank after our earlier cleaning pass - if they are,
# the row is unusable as a business record, since we can't join or
# aggregate without a valid key. We flag such rows rather than
# dropping them here, keeping that decision explicit for Gold.
#
# We deduplicate on booking_id (the natural business key) rather
# than comparing all 72 columns - a full-row comparison including
# wide text fields proved impractically slow at this row count.

df_step5 = df_step4.withColumn(
    "has_valid_keys",
    F.col("hotel_id").isNotNull()
    & F.col("booking_id").isNotNull()
    & F.col("customer_id").isNotNull()
)

print("Rows with valid keys:")
df_step5.groupBy("has_valid_keys").count().show()

before_dedup = df_step5.count()
df_step5 = df_step5.dropDuplicates(["booking_id"])
after_dedup = df_step5.count()

print(f"Rows before deduplication: {before_dedup:,}")
print(f"Rows after deduplication:  {after_dedup:,}")
print(f"Duplicate booking_id rows removed: {before_dedup - after_dedup:,}")

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 8, Finished, Available, Finished, False)

Rows with valid keys:
+--------------+------+
|has_valid_keys| count|
+--------------+------+
|          true|998595|
|         false| 52043|
+--------------+------+

Rows before deduplication: 1,050,638
Rows after deduplication:  1,050,638
Duplicate booking_id rows removed: 0


In [7]:
# Step 7 (revised): Split wide text columns into a separate table,
# and write a leaner main Silver table.
#
# review_text and review_title are large free-text fields rarely
# needed for analytical queries. Keeping them in the main table
# means every scan/write pays their I/O cost. We split them out
# into a narrow, joinable table instead.

REVIEW_TEXT_COLUMNS = ["review_text", "review_title"]

# Narrow table: just the key + the heavy text fields
df_reviews = df_step5.select("booking_id", *REVIEW_TEXT_COLUMNS)

# Main table: everything except the heavy text fields
df_silver_batch = df_step5.drop(*REVIEW_TEXT_COLUMNS)

print(f"Main table columns: {len(df_silver_batch.columns)}")
print(f"Reviews table columns: {len(df_reviews.columns)}")

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 9, Finished, Available, Finished, False)

Main table columns: 72
Reviews table columns: 3


In [8]:
# Step 8: Write both Silver tables.
#
# Main table: lean, analytics-ready booking data.
# Reviews table: narrow, joinable on booking_id when review text
# is actually needed.

df_silver_batch.write.format("delta").mode("overwrite").saveAsTable("silver_hotel_booking_batch")
print("Silver table 'silver_hotel_booking_batch' written successfully.")

df_reviews.write.format("delta").mode("overwrite").saveAsTable("silver_hotel_booking_reviews")
print("Silver table 'silver_hotel_booking_reviews' written successfully.")

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 10, Finished, Available, Finished, False)

Silver table 'silver_hotel_booking_batch' written successfully.
Silver table 'silver_hotel_booking_reviews' written successfully.


In [9]:
result_batch = spark.sql("SELECT COUNT(*) AS row_count FROM silver_hotel_booking_batch")
result_batch.show()

result_reviews = spark.sql("SELECT COUNT(*) AS row_count FROM silver_hotel_booking_reviews")
result_reviews.show()

spark.sql("""
    SELECT is_date_logic_valid, has_valid_keys, COUNT(*) AS count
    FROM silver_hotel_booking_batch
    GROUP BY is_date_logic_valid, has_valid_keys
    ORDER BY count DESC
""").show()

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 12, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|  1050638|
+---------+

+---------+
|row_count|
+---------+
|  1050638|
+---------+

+-------------------+--------------+------+
|is_date_logic_valid|has_valid_keys| count|
+-------------------+--------------+------+
|               true|          true|773662|
|              false|          true|116489|
|               NULL|          true|108444|
|               true|         false| 40247|
|              false|         false|  6055|
|               NULL|         false|  5741|
+-------------------+--------------+------+



In [10]:
# Step 9: Apply the same cleaning chain to the streaming Bronze table.
#
# Same schema, same cleaning functions - we're reusing everything
# we already defined (clean_placeholder, clean_numeric, clean_date,
# title_case, clean_boolean) rather than duplicating logic. The
# stream table is small, so performance isn't a concern here.

df_stream_bronze = spark.table("bronze_hotel_booking_stream")
print(f"Stream Bronze row count: {df_stream_bronze.count()}")

# Step 1: placeholder cleaning across all string columns
df_stream_step1 = df_stream_bronze
stream_string_columns = [f.name for f in df_stream_bronze.schema.fields if f.dataType.simpleString() == "string"]
for col_name in stream_string_columns:
    df_stream_step1 = df_stream_step1.withColumn(col_name, clean_placeholder(col_name))

# Step 2: type casting (numeric + date)
df_stream_step2 = (
    df_stream_step1
    .withColumn("total_rooms", clean_numeric("total_rooms", min_valid=1, max_valid=2000))
    .withColumn("star_rating", clean_numeric("star_rating", min_valid=1, max_valid=5))
    .withColumn("nights", clean_numeric("nights", min_valid=1, max_valid=90))
    .withColumn("adults", clean_numeric("adults", min_valid=0, max_valid=20))
    .withColumn("children", clean_numeric("children", min_valid=0, max_valid=20))
    .withColumn("infants", clean_numeric("infants", min_valid=0, max_valid=20))
    .withColumn("rooms_booked", clean_numeric("rooms_booked", min_valid=1, max_valid=50))
    .withColumn("total_price", clean_numeric("total_price", min_valid=0, max_valid=None))
    .withColumn("room_price", clean_numeric("room_price", min_valid=0, max_valid=None))
    .withColumn("tax_amount", clean_numeric("tax_amount", min_valid=0, max_valid=None))
    .withColumn("service_fee", clean_numeric("service_fee", min_valid=0, max_valid=None))
    .withColumn("paid_amount", clean_numeric("paid_amount", min_valid=0, max_valid=None))
    .withColumn("booking_date", clean_date("booking_date"))
    .withColumn("checkin_date", clean_date("checkin_date"))
    .withColumn("checkout_date", clean_date("checkout_date"))
)

# Step 3: cross-field date validation
df_stream_step3 = df_stream_step2.withColumn(
    "is_date_logic_valid",
    F.when(
        F.col("booking_date").isNull()
        | F.col("checkin_date").isNull()
        | F.col("checkout_date").isNull(),
        None,
    )
    .when(
        (F.col("checkin_date") < F.col("booking_date"))
        | (F.col("checkout_date") < F.col("checkin_date")),
        False,
    )
    .otherwise(True)
)

# Step 4: casing + boolean normalization
df_stream_step4 = (
    df_stream_step3
    .withColumn("country", title_case("country"))
    .withColumn("city", title_case("city"))
    .withColumn("payment_status", title_case("payment_status"))
    .withColumn("payment_method", title_case("payment_method"))
    .withColumn("booking_status", title_case("booking_status"))
    .withColumn("is_cancelled", clean_boolean("is_cancelled"))
)

# Step 5: key validation + dedup on booking_id
df_stream_step5 = df_stream_step4.withColumn(
    "has_valid_keys",
    F.col("hotel_id").isNotNull()
    & F.col("booking_id").isNotNull()
    & F.col("customer_id").isNotNull()
).dropDuplicates(["booking_id"])

print(f"Stream row count after cleaning: {df_stream_step5.count()}")

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 14, Finished, Available, Finished, False)

Stream Bronze row count: 952
Stream row count after cleaning: 952


In [11]:
# Step 10: Split review columns from the stream data (same pattern
# as batch), then union both sources into a single Silver table.

df_stream_reviews = df_stream_step5.select("booking_id", *REVIEW_TEXT_COLUMNS)
df_stream_silver = df_stream_step5.drop(*REVIEW_TEXT_COLUMNS)

print(f"Batch columns:  {len(df_silver_batch.columns)}")
print(f"Stream columns: {len(df_stream_silver.columns)}")

# Union requires identical schemas - let's verify before combining
assert set(df_silver_batch.columns) == set(df_stream_silver.columns), \
    "Schema mismatch between batch and stream - cannot union safely"

df_silver_hotel_booking = df_silver_batch.unionByName(df_stream_silver)
df_silver_reviews_combined = df_reviews.unionByName(df_stream_reviews)

print(f"Combined booking row count: {df_silver_hotel_booking.count():,}")
print(f"Combined reviews row count: {df_silver_reviews_combined.count():,}")

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 15, Finished, Available, Finished, False)

Batch columns:  72
Stream columns: 72
Combined booking row count: 1,051,590
Combined reviews row count: 1,051,590


In [12]:
# Step 11: Write the final, combined Silver tables.
#
# These represent "one clean version of the truth" regardless of
# whether a record originally arrived via batch or streaming
# ingestion - the defining goal of the Silver layer in a medallion
# architecture.

df_silver_hotel_booking.write.format("delta").mode("overwrite").saveAsTable("silver_hotel_booking")
print("Silver table 'silver_hotel_booking' written successfully.")

df_silver_reviews_combined.write.format("delta").mode("overwrite").saveAsTable("silver_hotel_booking_reviews_combined")
print("Silver table 'silver_hotel_booking_reviews_combined' written successfully.")

StatementMeta(, 83a06bfb-9ff4-4f16-8092-5e30cb15e11f, 16, Finished, Available, Finished, False)

Silver table 'silver_hotel_booking' written successfully.
Silver table 'silver_hotel_booking_reviews_combined' written successfully.
